# 🚬 煙霧偵測模型訓練 — Google Colab

**資料集：** [Roboflow smoke-gxoy3](https://universe.roboflow.com/naruesuan-university/smoke-gxoy3)  
**模型：** YOLOv8n（輕量，適合邊緣部署）  
**目標：** 訓練後匯出 ONNX 模型，複製到 `models/smoke_detector.onnx` 啟用條件 3

## 開始前請確認
- 上方選單：`執行階段 → 變更執行階段類型 → GPU (T4)`
- 準備好 Roboflow API Key（[取得方式](https://app.roboflow.com/) → Settings → Roboflow API）

## 0. 確認 GPU

In [ ]:
!nvidia-smi

## 1. 安裝套件

In [ ]:
!pip install -q roboflow ultralytics onnx onnxsim

## 2. 下載煙霧資料集

輸入你的 Roboflow API Key：

In [ ]:
import yaml, random, shutil, glob, os

MERGED_DIR = "/content/smoke_merged"

# ── 從合併的 train 切出 10% 作為 val ───────────────────────────────────────
all_imgs = sorted(
    glob.glob(f"{MERGED_DIR}/images/train/*.jpg") +
    glob.glob(f"{MERGED_DIR}/images/train/*.png")
)
random.seed(42)
random.shuffle(all_imgs)
split = max(10, int(len(all_imgs) * 0.1))

val_img_dir = f"{MERGED_DIR}/images/val"
val_lbl_dir = f"{MERGED_DIR}/labels/val"
os.makedirs(val_img_dir, exist_ok=True)
os.makedirs(val_lbl_dir, exist_ok=True)

for img_path in all_imgs[:split]:
    fname = os.path.basename(img_path)
    stem  = os.path.splitext(fname)[0]
    shutil.move(img_path, f"{val_img_dir}/{fname}")
    lbl_src = f"{MERGED_DIR}/labels/train/{stem}.txt"
    if os.path.exists(lbl_src):
        shutil.move(lbl_src, f"{val_lbl_dir}/{stem}.txt")

n_train = len(glob.glob(f"{MERGED_DIR}/images/train/*.jpg") +
               glob.glob(f"{MERGED_DIR}/images/train/*.png"))
n_val   = len(glob.glob(f"{val_img_dir}/*.jpg") + glob.glob(f"{val_img_dir}/*.png"))
print(f"✅ 合併資料集：train={n_train} 張，val={n_val} 張")

# ── 建立合併資料集的 data.yaml ──────────────────────────────────────────────
merged_yaml = f"{MERGED_DIR}/data.yaml"
cfg = {
    "path":  MERGED_DIR,
    "train": "images/train",
    "val":   "images/val",
    "nc":    1,
    "names": ["smoke"],
}
with open(merged_yaml, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

print(f"✅ data.yaml 已寫入：{merged_yaml}")
!cat /content/smoke_merged/data.yaml

In [ ]:
import os, shutil, glob, random
from roboflow import Roboflow

# ── 額外資料集清單 ──────────────────────────────────────────────────────────
# 每個 entry: (workspace, project)，自動抓最新版本
EXTRA_DATASETS = [
    ("robmarkcole",   "smoke-detection-cctv"),
    ("teamwork-ewwzs","smoke-dataset-6g0yo"),
    ("shresthab",     "smoke-detector-qrjh9"),
]

MERGED_DIR = "/content/smoke_merged"

def get_latest_version(project):
    """取得 project 最新可用版本"""
    versions = project.versions()
    if not versions:
        return None
    return versions[-1]

def copy_split(src_base, dst_base, split_map):
    """
    將 src_base 下的影像/標籤複製到 dst_base/images/train 與 dst_base/labels/train
    split_map: {'images': 來源影像相對路徑, 'labels': 來源標籤相對路徑}
    """
    for kind in ("images", "labels"):
        src_dir = os.path.join(src_base, split_map[kind])
        if not os.path.exists(src_dir):
            print(f"  ⚠️  找不到 {src_dir}，跳過")
            continue
        dst_dir = os.path.join(dst_base, kind, "train")
        os.makedirs(dst_dir, exist_ok=True)
        files = glob.glob(f"{src_dir}/*")
        for f in files:
            dst_f = os.path.join(dst_dir, os.path.basename(f))
            # 避免檔名衝突，加上資料集前綴
            if os.path.exists(dst_f):
                base, ext = os.path.splitext(os.path.basename(f))
                dst_f = os.path.join(dst_dir, f"{base}_{random.randint(1000,9999)}{ext}")
            shutil.copy2(f, dst_f)
        print(f"  ✅ 複製 {kind}: {len(files)} 個檔案")

def detect_split_dirs(base):
    """自動偵測影像與標籤所在子路徑"""
    candidates_img = ["train/images", "images/train", "train"]
    candidates_lbl = ["train/labels", "labels/train", "train/labels"]
    img_rel = next((c for c in candidates_img
                    if glob.glob(f"{base}/{c}/*.jpg") or glob.glob(f"{base}/{c}/*.png")), None)
    lbl_rel = next((c for c in candidates_lbl
                    if glob.glob(f"{base}/{c}/*.txt")), None)
    return img_rel, lbl_rel

# ── Step 1：複製主資料集（已下載的 smoke-gxoy3）────────────────────────────
os.makedirs(MERGED_DIR, exist_ok=True)
print("=== 複製主資料集 smoke-gxoy3 ===")
main_base = "/content/smoke_dataset"
img_rel, lbl_rel = detect_split_dirs(main_base)
if img_rel and lbl_rel:
    copy_split(main_base, MERGED_DIR, {"images": img_rel, "labels": lbl_rel})
else:
    print("⚠️  主資料集路徑偵測失敗，請確認 Cell 2 已執行")

# ── Step 2：下載並合併額外資料集 ────────────────────────────────────────────
rf = Roboflow(api_key=API_KEY)
failed = []

for ws, proj_name in EXTRA_DATASETS:
    print(f"\n=== 下載 {ws}/{proj_name} ===")
    try:
        project = rf.workspace(ws).project(proj_name)
        version = get_latest_version(project)
        if version is None:
            print(f"  ⚠️  找不到可用版本，跳過")
            failed.append(f"{ws}/{proj_name}")
            continue
        print(f"  使用版本：{version.version}")
        dl_dir = f"/content/extra_{proj_name}"
        version.download("yolov8", location=dl_dir)

        img_rel, lbl_rel = detect_split_dirs(dl_dir)
        if img_rel and lbl_rel:
            copy_split(dl_dir, MERGED_DIR, {"images": img_rel, "labels": lbl_rel})
        else:
            print(f"  ⚠️  無法偵測目錄結構，跳過合併")
            failed.append(f"{ws}/{proj_name}")
    except Exception as e:
        print(f"  ❌ 下載失敗：{e}")
        failed.append(f"{ws}/{proj_name}")

# ── 統計結果 ────────────────────────────────────────────────────────────────
total_imgs = (glob.glob(f"{MERGED_DIR}/images/train/*.jpg") +
              glob.glob(f"{MERGED_DIR}/images/train/*.png"))
total_lbls = glob.glob(f"{MERGED_DIR}/labels/train/*.txt")

print(f"\n{'='*50}")
print(f"合併完成：影像 {len(total_imgs)} 張，標籤 {len(total_lbls)} 個")
if failed:
    print(f"⚠️  以下資料集下載失敗（不影響訓練）：{failed}")
print(f"{'='*50}")

## 2b. 下載額外煙霧資料集並合併

合併以下公開資料集以提升訓練資料量（預計總量 2000~3500 張）：

| 資料集 | 來源 | 說明 |
|--------|------|------|
| smoke-gxoy3 | naruesuan-university | 主資料集（已下載） |
| smoke-detection | robmarkcole | 室內外煙霧 |
| forest-fire-and-smoke | various | 山火煙霧 |
| smoke-and-fire | abdullahtariq | 多場景煙霧 |

In [ ]:
from roboflow import Roboflow

API_KEY = "YOUR_ROBOFLOW_API_KEY"  # ← 替換為你的 Key

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("naruesuan-university").project("smoke-gxoy3")

# 自動列出並使用最新可用版本（避免指定版本號不存在的問題）
versions = project.versions()
print(f"可用版本號：{[v.version for v in versions]}")
latest = versions[-1]
print(f"使用版本：{latest.version}")

dataset = latest.download("yolov8", location="/content/smoke_dataset")
print(f"資料集路徑：{dataset.location}")

## 3. 確認資料集結構與統計

In [ ]:
# 列出原始 data.yaml 供參考
!cat /content/smoke_dataset/data.yaml

import torch
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else "cpu"
print(f"使用裝置：{'GPU (' + torch.cuda.get_device_name(0) + ')' if device == 0 else 'CPU'}")

# 使用合併資料集（若合併失敗則退回單一資料集）
import os
DATA_YAML = ("/content/smoke_merged/data.yaml"
             if os.path.exists("/content/smoke_merged/data.yaml")
             else "/content/smoke_dataset/data.yaml")
print(f"資料集：{DATA_YAML}")

model = YOLO("yolov8s.pt")

results = model.train(
    data=DATA_YAML,
    epochs=100,
    batch=16 if device == 0 else 4,
    imgsz=640,
    device=device,
    project="/content/runs",
    name="smoke_detector",
    patience=20,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    save=True,
    save_period=10,
    exist_ok=True,
)

In [ ]:
import yaml, os, glob, shutil, random

base = "/content/smoke_dataset"

# 印出實際目錄結構
print("=== 實際目錄結構 ===")
for root, dirs, files in os.walk(base):
    depth = root.replace(base, "").count(os.sep)
    if depth > 2:
        continue
    print("  " * depth + os.path.basename(root) + "/")

# 偵測 train images
def find_images_dir(base, candidates):
    for c in candidates:
        p = os.path.join(base, c)
        imgs = glob.glob(f"{p}/*.jpg") + glob.glob(f"{p}/*.png") + glob.glob(f"{p}/*.jpeg")
        if imgs:
            return c, sorted(imgs)
    return None, []

train_rel, train_imgs = find_images_dir(base, ["train/images", "images/train", "train"])
val_rel,   val_imgs   = find_images_dir(base, ["valid/images", "val/images", "images/val", "valid", "val"])

assert train_rel and train_imgs, "❌ 找不到 train 影像目錄"

# 若沒有 val，自動從 train 切出 10%
if not val_rel:
    print(f"\n⚠️  未找到驗證集，從 train ({len(train_imgs)} 張) 自動切割 10% 作為 val ...")
    random.seed(42)
    random.shuffle(train_imgs)
    split = max(1, int(len(train_imgs) * 0.1))
    val_imgs_to_move   = train_imgs[:split]
    train_imgs_to_keep = train_imgs[split:]

    # 建立 val 目錄
    val_img_dir = os.path.join(base, "valid/images")
    val_lbl_dir = os.path.join(base, "valid/labels")
    os.makedirs(val_img_dir, exist_ok=True)
    os.makedirs(val_lbl_dir, exist_ok=True)

    train_lbl_dir = os.path.join(base, train_rel.replace("images", "labels"))

    for img_path in val_imgs_to_move:
        fname = os.path.basename(img_path)
        stem  = os.path.splitext(fname)[0]
        # 移動影像
        shutil.move(img_path, os.path.join(val_img_dir, fname))
        # 移動對應標籤（若存在）
        lbl_src = os.path.join(train_lbl_dir, stem + ".txt")
        if os.path.exists(lbl_src):
            shutil.move(lbl_src, os.path.join(val_lbl_dir, stem + ".txt"))

    val_rel = "valid/images"
    print(f"✅ 切割完成：train={len(train_imgs_to_keep)} 張，val={split} 張")
else:
    print(f"\n✅ train={len(train_imgs)} 張，val={len(val_imgs)} 張")

# 更新 data.yaml
yaml_path = f"{base}/data.yaml"
with open(yaml_path) as f:
    cfg = yaml.safe_load(f)

cfg["path"]  = base
cfg["train"] = train_rel
cfg["val"]   = val_rel
cfg["nc"]    = 1
cfg["names"] = ["smoke"]

with open(yaml_path, "w") as f:
    yaml.dump(cfg, f, allow_unicode=True, default_flow_style=False)

print("\n=== 修正後 data.yaml ===")
!cat /content/smoke_dataset/data.yaml

## 5. 訓練模型

T4 GPU 大約需要 **20–40 分鐘**（100 epochs, batch=16）

In [ ]:
import torch
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else "cpu"
print(f"使用裝置：{'GPU (' + torch.cuda.get_device_name(0) + ')' if device == 0 else 'CPU（建議切換至 GPU 執行階段）'}")

model = YOLO("yolov8s.pt")  # yolov8s：比 n 大 2 倍，準確度明顯提升

results = model.train(
    data="/content/smoke_dataset/data.yaml",
    epochs=100,
    batch=16 if device == 0 else 4,
    imgsz=640,
    device=device,
    project="/content/runs",
    name="smoke_detector",
    patience=20,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    save=True,
    save_period=10,
    exist_ok=True,
)

## 6. 查看訓練結果

In [ ]:
from IPython.display import Image as IPImage, display
import glob

# 訓練曲線
plots = glob.glob("/content/runs/smoke_detector/*.png")
for p in sorted(plots):
    display(IPImage(p))

In [ ]:
# 驗證最終指標
import glob, os

# 自動找最新訓練結果（相容重複執行時的 smoke_detector2 等命名）
candidates = sorted(glob.glob("/content/runs/smoke_detector*/weights/best.pt"))
assert candidates, "❌ 找不到 best.pt，請確認訓練已完成"
best_pt = candidates[-1]
print(f"使用模型：{best_pt}")

eval_model = YOLO(best_pt)
metrics = eval_model.val(data="/content/smoke_dataset/data.yaml")
print(f"\nmAP50    : {metrics.box.map50:.4f}")
print(f"mAP50-95 : {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall   : {metrics.box.mr:.4f}")

## 7. 匯出 ONNX 模型

In [ ]:
export_model = YOLO(best_pt)
export_model.export(
    format="onnx",
    imgsz=640,
    simplify=True,
    opset=17,
)

onnx_path = best_pt.replace(".pt", ".onnx")
print(f"\n✅ ONNX 模型位於：{onnx_path}")

## 8. 下載模型到本機

執行後會自動下載 `smoke_detector.onnx`，下載完成後複製到專案的 `models/` 目錄：

In [ ]:
import shutil
from google.colab import files

# 複製為統一命名
dst = "/content/smoke_detector.onnx"
shutil.copy(onnx_path, dst)
print(f"檔案大小：{os.path.getsize(dst)/1024/1024:.1f} MB")

# 下載到本機
files.download(dst)
print("\n📦 下載完成後，將 smoke_detector.onnx 複製到專案的 models/ 目錄")
print("   路徑：cgr_detection/models/smoke_detector.onnx")
print("   重啟推論系統即自動啟用條件 3（煙霧偵測）")

## 9. （選用）同時下載 .pt 權重備份

In [ ]:
# 如果想保留 PyTorch 權重以便日後繼續訓練
files.download(best_pt)

---

## 完成後的部署步驟

```bash
# 1. 將下載的模型放到專案目錄
cp ~/Downloads/smoke_detector.onnx /path/to/cgr_detection/models/

# 2. 重啟推論系統（自動偵測並啟用條件 3）
python infer_main.py
```

啟動時會看到：
```
[func] 煙霧偵測模型已啟用（條件 3）
```

條件 3 觸發時畫面顯示橘色煙霧框與 `[+10 Smoke]` 計分提示。